# Retail Analytics — Combined Cleaning + ML Pipeline

**One notebook. One Flask server. One ngrok tunnel. No conflicts.**

**Run ALL cells top to bottom every time you open Colab.**

| Cell | What it does |
|------|--------------|
| 1 | Install packages + imports |
| 2 | Mount Google Drive |
| 3 | clean_file() + run_cleaning() — universal cleaning function |
| 4 | optimise_dtypes() + run_ml() — full ML pipeline |
| 5 | Optional manual test |
| 6 | Combined Flask server + ngrok (one server for both) |
| 7 | Keep-alive |

**Endpoints n8n uses:**
```
POST /run-cleaning    → trigger cleaning
GET  /cleaning-status → poll cleaning result
POST /run-pipeline    → trigger ML
GET  /status          → poll ML result
GET  /ping            → health check
```

**n8n workflow stays exactly the same — just use one URL for all nodes.**

In [ ]:
# ============================================================
# CELL 1 — Install packages & imports
# ============================================================

!pip install xgboost flask pyngrok -q

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, warnings, os, json, gc, re
from datetime import date

from sklearn.model_selection import train_test_split
from sklearn.linear_model    import LogisticRegression
from sklearn.metrics         import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing   import StandardScaler
from xgboost                 import XGBClassifier

warnings.filterwarnings('ignore')
print('All packages ready!')

All packages ready!


In [ ]:
# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Retail_Pipeline'
RAW  = f'{BASE}/Raw_Data'
CLN  = f'{BASE}/cleaned_data'
OUT  = f'{BASE}/ML_Output'
MDL  = f'{BASE}/models'

for folder in [CLN, OUT, MDL]:
    os.makedirs(folder, exist_ok=True)

# Verify raw CSV files
csv_files = sorted([f for f in os.listdir(RAW) if f.endswith('.csv')])
print(f'\nGoogle Drive mounted!')
print(f'Raw data:     {RAW}')
print(f'Cleaned data: {CLN}')
print(f'ML Output:    {OUT}')
print(f'\nFound {len(csv_files)} CSV files in Raw_Data:')
for f in csv_files:
    size = f'{os.path.getsize(os.path.join(RAW, f))/1e6:.1f} MB'
    print(f'  {f:40s} {size}')

Mounted at /content/drive

Google Drive mounted!
Raw data:     /content/drive/MyDrive/Retail_Pipeline/Raw_Data
Cleaned data: /content/drive/MyDrive/Retail_Pipeline/cleaned_data
ML Output:    /content/drive/MyDrive/Retail_Pipeline/ML_Output

Found 7 CSV files in Raw_Data:
  Date_Dimension.csv                       0.0 MB
  Stores.csv                               0.0 MB
  aisles.csv                               0.0 MB
  departments.csv                          0.0 MB
  order_products.csv                       577.6 MB
  orders.csv                               109.0 MB
  products.csv                             2.2 MB


In [ ]:
# ============================================================
# CELL 3 — Universal Cleaning Functions
#
# Cleaning steps applied to every CSV found in Raw_Data/:
#   1. Standardise column names
#   2. Strip whitespace from text columns
#   3. Remove fully empty rows
#   4. Remove exact duplicate rows
#   5. Composite key deduplication (auto-detected)
#   6. Range checks — remove invalid values (auto-detected)
#   7. Fill nulls with 0 (except days_since_prior_order)
#   8. Remove outliers in ML feature columns (IQR x1.5)
# ============================================================

RANGE_RULES = {
    'order_dow':           (0, 6),
    'order_hour_of_day':   (0, 23),
    'reordered':           (0, 1),
    'add_to_cart_order':   (1, 9999),
}

COMPOSITE_KEY_RULES = [
    ['order_id', 'product_id'],
]

# days_since_prior_order: null = first ever order = valid
# Filling with 0 would mean ordered 0 days ago which is wrong
KEEP_NULLS = ['days_since_prior_order']

# Outlier removal only on ML feature columns
OUTLIER_COLS = [
    'times_purchased', 'purchase_rate', 'product_total_orders',
    'total_orders', 'avg_days_between_orders', 'avg_order_hour',
]


def standardise_columns(df):
    df.columns = [
        re.sub(r'[^a-z0-9]+', '_', col.strip().lower()).strip('_')
        for col in df.columns
    ]
    return df


def clean_file(df, filename):
    original_rows = len(df)
    report = {
        'file': filename, 'rows_before': original_rows,
        'columns': list(df.columns), 'actions': [], 'warnings': []
    }

    # Step 1: Standardise column names
    original_cols = list(df.columns)
    df = standardise_columns(df)
    renamed = [f'{o} -> {n}' for o, n in zip(original_cols, df.columns) if o != n]
    if renamed:
        report['actions'].append(f'Columns renamed: {renamed}')
    report['columns'] = list(df.columns)

    # Step 2: Strip whitespace
    str_cols = df.select_dtypes(include='object').columns.tolist()
    for col in str_cols:
        df[col] = df[col].str.strip()
    if str_cols:
        report['actions'].append(f'Whitespace stripped from {len(str_cols)} columns')

    # Step 3: Remove fully empty rows
    before = len(df)
    df = df.dropna(how='all')
    if before - len(df) > 0:
        report['actions'].append(f'Fully empty rows removed: {before-len(df):,}')

    # Step 4: Remove exact duplicates
    before = len(df)
    df = df.drop_duplicates()
    report['actions'].append(f'Exact duplicate rows removed: {before-len(df):,}')

    # Step 5: Composite key deduplication
    for key_cols in COMPOSITE_KEY_RULES:
        if all(c in df.columns for c in key_cols):
            before = len(df)
            df = df.drop_duplicates(subset=key_cols)
            report['actions'].append(
                f'Composite key {key_cols} duplicates removed: {before-len(df):,}'
            )

    # Step 6: Range checks
    for col, (lo, hi) in RANGE_RULES.items():
        if col in df.columns:
            before = len(df)
            df = df[df[col].between(lo, hi, inclusive='both') | df[col].isnull()]
            report['actions'].append(
                f'{col} outside [{lo}-{hi}] removed: {before-len(df):,}'
            )

    # Step 7: Fill nulls with 0 except KEEP_NULLS
    null_before = df.isnull().sum()
    cols_to_fill = [c for c in df.columns if c not in KEEP_NULLS]
    df[cols_to_fill] = df[cols_to_fill].fillna(0)
    for col in cols_to_fill:
        if null_before.get(col, 0) > 0:
            report['actions'].append(
                f'{col}: {null_before[col]:,} nulls filled with 0'
            )
    for col in KEEP_NULLS:
        if col in df.columns and df[col].isnull().sum() > 0:
            n = df[col].isnull().sum()
            report['actions'].append(
                f'{col}: {n:,} nulls KEPT — valid first-order rows'
            )

    # Step 8: Outlier removal on ML feature columns only
    for col in OUTLIER_COLS:
        if col not in df.columns:
            continue
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            continue
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        before = len(df)
        df = df[df[col].between(lo, hi) | df[col].isnull()]
        removed = before - len(df)
        if removed > 0:
            report['actions'].append(
                f'{col} outliers removed (IQR x1.5): {removed:,}'
            )

    report['rows_after']   = len(df)
    report['rows_removed'] = original_rows - len(df)
    return df, report


def run_cleaning():
    today   = date.today().isoformat()
    reports = []

    print('=' * 60)
    print('UNIVERSAL DATA CLEANING PIPELINE')
    print('=' * 60)

    csv_files = sorted([f for f in os.listdir(RAW) if f.endswith('.csv')])
    if not csv_files:
        raise FileNotFoundError(f'No CSV files found in {RAW}')

    print(f'\nFound {len(csv_files)} CSV files to clean:')
    for f in csv_files:
        print(f'  - {f}')

    for i, filename in enumerate(csv_files, 1):
        print(f'\n[{i}/{len(csv_files)}] Cleaning {filename} ...')
        filepath     = os.path.join(RAW, filename)
        file_size_mb = os.path.getsize(filepath) / 1e6

        # Large file: dtype optimisation to prevent RAM crash
        if file_size_mb > 50:
            print(f'  Large file ({file_size_mb:.0f} MB) — dtype optimisation...')
            df = pd.read_csv(filepath)
            for col in df.select_dtypes('int64').columns:
                df[col] = pd.to_numeric(df[col], downcast='integer')
            for col in df.select_dtypes('float64').columns:
                df[col] = pd.to_numeric(df[col], downcast='float')
        else:
            df = pd.read_csv(filepath)

        print(f'  Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
        df, rep = clean_file(df, filename)

        output_path = os.path.join(CLN, filename)
        df.to_csv(output_path, index=False)
        # Force flush to Google Drive for large files
        # Without this, orders.csv and order_products.csv
        # may not fully write before the function returns
        if file_size_mb > 50:
            import subprocess
            subprocess.run(['sync'], check=False)
            print(f'  Flushed to Drive: {filename}')
        rep['saved_as'] = filename
        reports.append(rep)

        print(f'  Rows: {rep["rows_before"]:,} -> {rep["rows_after"]:,} '
              f'(removed {rep["rows_removed"]:,})')
        print(f'  Saved: cleaned_data/{filename}')
        for action in rep['actions']:
            print(f'    - {action}')

    total_before  = sum(r['rows_before']  for r in reports)
    total_after   = sum(r['rows_after']   for r in reports)
    total_removed = total_before - total_after

    print('\n' + '=' * 60)
    print('CLEANING SUMMARY')
    print('=' * 60)
    for r in reports:
        status = 'SAME' if r['rows_removed'] == 0 else f'-{r["rows_removed"]:,}'
        print(f'  {r["file"]:35s} {r["rows_before"]:>12,} rows  [{status}]')
    print(f'\n  Total files cleaned:    {len(reports)}')
    print(f'  Total rows removed:     {total_removed:,}')
    print(f'  Saved to:               {CLN}')
    print('=' * 60)
    print('ALL FILES CLEANED AND SAVED')

    return {
        'status':        'success',
        'date':          today,
        'files_cleaned': len(reports),
        'total_removed': int(total_removed),
        'summary': [
            {
                'file':         r['file'],
                'rows_before':  int(r['rows_before']),
                'rows_after':   int(r['rows_after']),
                'rows_removed': int(r['rows_removed']),
                'columns':      r['columns'],
                'saved_as':     r['saved_as'],
                'actions':      r['actions']
            }
            for r in reports
        ]
    }


print('Cleaning functions defined and ready!')

Cleaning functions defined and ready!


In [ ]:
# ============================================================
# CELL 4 — ML Pipeline Function
#
# Steps:
#   1.  Load data from Google Drive (cleaned + raw)
#   2.  Merge order metadata onto order_products
#   3.  Train/future split using order_number (leakage-free)
#   4.  Feature engineering (6 features)
#   5.  Build master dataset
#   6.  Stratified sample (500k rows)
#   7a. Train Logistic Regression baseline
#   7b. Train XGBoost main model
#   8.  Generate reorder probabilities (batched)
#   9.  Generate business insights
#   10. Save all outputs + charts to Google Drive
# ============================================================

def optimise_dtypes(df):
    for col in df.select_dtypes('int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes('float64').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    return df


def run_ml():
    today = date.today().isoformat()

    print('STEP 1: Loading data from Google Drive...')
    orders         = optimise_dtypes(pd.read_csv(f'{CLN}/orders.csv'))
    order_products = optimise_dtypes(pd.read_csv(f'{RAW}/order_products.csv'))
    products       = pd.read_csv(f'{RAW}/products.csv')
    aisles         = pd.read_csv(f'{RAW}/aisles.csv')
    departments    = pd.read_csv(f'{RAW}/departments.csv')
    print(f'  orders:  {orders.shape}')
    print(f'  order_products:  {order_products.shape}')

    print('\nSTEP 2: Merging...')
    op = order_products.merge(
        orders[['order_id', 'user_id', 'order_number',
                'order_dow', 'order_hour_of_day', 'days_since_prior_order']],
        on='order_id', how='left'
    )
    orders_full         = orders.copy()
    order_products_full = order_products.copy()
    del order_products, orders
    gc.collect()
    print(f'  Merged shape: {op.shape}  |  RAM freed')

    print('\nSTEP 3: Train/future split...')
    last_order_num = op.groupby('user_id')['order_number'].max().reset_index()
    last_order_num.columns = ['user_id', 'max_order_number']
    op = op.merge(last_order_num, on='user_id', how='left')
    history = op[op['order_number'] < op['max_order_number']].copy()
    last    = op[op['order_number'] == op['max_order_number']].copy()
    del op, last_order_num
    gc.collect()
    print(f'  History rows:    {len(history):,}')
    print(f'  Last order rows: {len(last):,}  |  RAM freed')

    print('\nSTEP 4: Engineering features...')
    customer_features = history.groupby('user_id').agg(
        total_orders            = ('order_number', 'max'),
        avg_days_between_orders = ('days_since_prior_order', 'mean'),
        avg_order_hour          = ('order_hour_of_day', 'mean')
    ).reset_index()
    product_features = history.groupby('product_id').agg(
        product_total_orders = ('order_id',  'count'),
        product_reorder_rate = ('reordered', 'mean')
    ).reset_index()
    user_product = history.groupby(['user_id', 'product_id']).agg(
        times_purchased = ('order_id', 'count')
    ).reset_index()
    last_items = last[['user_id', 'product_id']].drop_duplicates().copy()
    last_items['target_reorder'] = 1
    user_product = user_product.merge(last_items, on=['user_id', 'product_id'], how='left')
    user_product['target_reorder'] = user_product['target_reorder'].fillna(0).astype(int)
    del history, last, last_items
    gc.collect()
    print(f'  User-product pairs: {len(user_product):,}')

    print('\nSTEP 5: Building master dataset...')
    dataset = user_product.merge(customer_features, on='user_id',    how='left')
    dataset = dataset.merge(product_features,       on='product_id', how='left')
    dataset = dataset.merge(
        products[['product_id', 'product_name', 'aisle_id', 'department_id']],
        on='product_id', how='left'
    )
    dataset = dataset.merge(aisles,      on='aisle_id',      how='left')
    dataset = dataset.merge(departments, on='department_id', how='left')
    dataset.fillna(0, inplace=True)
    dataset['purchase_rate'] = dataset['times_purchased'] / (dataset['total_orders'] + 1)
    del user_product, customer_features, product_features
    gc.collect()
    print(f'  Dataset shape: {dataset.shape}  |  RAM freed')

    print('\nSTEP 6: Stratified sample (500k rows)...')
    majority = dataset[dataset['target_reorder'] == 0]
    minority = dataset[dataset['target_reorder'] == 1]
    n_minority = min(len(minority), 50_000)
    n_majority = min(len(majority), 450_000)
    majority_sample = majority.sample(n=n_majority, random_state=42)
    minority_sample = minority.sample(n=n_minority, random_state=42)
    dataset_sample  = pd.concat([majority_sample, minority_sample]).sample(
        frac=1, random_state=42
    ).reset_index(drop=True)
    print(f'  Sample: {dataset_sample.shape}  majority={n_majority:,}  minority={n_minority:,}')

    features = [f for f in [
        'times_purchased', 'purchase_rate', 'product_total_orders',
        'total_orders', 'avg_days_between_orders', 'avg_order_hour'
    ] if f in dataset_sample.columns]

    X = dataset_sample[features].fillna(0)
    y = dataset_sample['target_reorder']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print('\nSTEP 7a: Training Logistic Regression baseline...')
    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)
    lr_model = LogisticRegression(max_iter=500, random_state=42)
    lr_model.fit(X_train_scaled, y_train)
    lr_preds = lr_model.predict(X_test_scaled)
    lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]
    lr_acc   = round(float(accuracy_score(y_test, lr_preds)), 4)
    lr_auc   = round(float(roc_auc_score(y_test, lr_proba)), 4)
    print(f'  LR  - Accuracy: {lr_acc}  |  AUC-ROC: {lr_auc}')

    print('\nSTEP 7b: Training XGBoost (main model)...')
    scale_pos_weight = n_majority / n_minority
    xgb_model = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=scale_pos_weight, subsample=0.8,
        colsample_bytree=0.8, min_child_weight=5,
        random_state=42, n_jobs=-1, eval_metric='logloss', verbosity=0
    )
    xgb_model.fit(X_train, y_train)
    xgb_preds = xgb_model.predict(X_test)
    xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
    xgb_acc   = round(float(accuracy_score(y_test, xgb_preds)), 4)
    xgb_auc   = round(float(roc_auc_score(y_test, xgb_proba)), 4)
    print(f'  XGB - Accuracy: {xgb_acc}  |  AUC-ROC: {xgb_auc}')
    print(classification_report(y_test, xgb_preds))

    feat_importance = dict(zip(
        features, [round(float(v), 4) for v in xgb_model.feature_importances_]
    ))

    y_test_store    = y_test.copy()
    lr_preds_store  = lr_preds.copy()
    xgb_preds_store = xgb_preds.copy()
    del dataset_sample, majority, minority, majority_sample, minority_sample
    del X, y, X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled
    gc.collect()

    print('\nSTEP 8: Generating reorder probabilities (batched)...')
    BATCH  = 500_000
    X_full = dataset[features].fillna(0)
    probs  = []
    for start in range(0, len(X_full), BATCH):
        probs.append(xgb_model.predict_proba(X_full.iloc[start:start+BATCH])[:, 1])
        print(f'  ... {min(start+BATCH, len(X_full)):,} / {len(X_full):,}')
    dataset['reorder_probability'] = np.concatenate(probs).round(4)
    del X_full, probs
    gc.collect()

    print('\nSTEP 9: Generating business insights...')
    product_demand = order_products_full.groupby('product_id').agg(
        total_orders = ('order_id',  'count'),
        reorder_rate = ('reordered', 'mean')
    ).reset_index().merge(products, on='product_id', how='left')

    top_products = product_demand.sort_values(
        'total_orders', ascending=False
    ).head(10).round(4)

    inventory = product_demand[product_demand['total_orders'] >= 100].copy()
    inventory['demand_norm'] = (
        inventory['total_orders'] - inventory['total_orders'].min()
    ) / (inventory['total_orders'].max() - inventory['total_orders'].min())
    avg_reorder_prob = (
        dataset.groupby('product_id')['reorder_probability'].mean().reset_index()
    )
    avg_reorder_prob.columns = ['product_id', 'avg_reorder_prob']
    inventory = inventory.merge(avg_reorder_prob, on='product_id', how='left')
    inventory['avg_reorder_prob'] = inventory['avg_reorder_prob'].fillna(0)
    inventory['final_priority'] = (
        0.4 * inventory['demand_norm'] +
        0.3 * inventory['reorder_rate'] +
        0.3 * inventory['avg_reorder_prob']
    ).round(4)
    inventory['priority_tier'] = pd.cut(
        inventory['final_priority'], bins=[0, 0.33, 0.66, 1.0],
        labels=['Low', 'Medium', 'High']
    )
    inventory = inventory.sort_values('final_priority', ascending=False)

    print('\nSTEP 10: Saving outputs to Google Drive...')
    pred_file = f'reorder_predictions_{today}.csv'
    dataset.to_csv(f'{OUT}/{pred_file}', index=False)
    dataset.to_csv(f'{OUT}/reorder_predictions_latest.csv', index=False)
    print(f'  Predictions saved: {pred_file}')

    inv_file = f'inventory_priority_{today}.csv'
    inventory.to_csv(f'{OUT}/{inv_file}', index=False)
    inventory.to_csv(f'{OUT}/inventory_priority_latest.csv', index=False)
    print(f'  Inventory saved:   {inv_file}')

    top_file = f'top_products_{today}.csv'
    top_products.to_csv(f'{OUT}/{top_file}', index=False)
    top_products.to_csv(f'{OUT}/top_products_latest.csv', index=False)
    print(f'  Top products saved: {top_file}')

    # Charts
    print('  Saving charts...')
    top10 = product_demand.sort_values('total_orders', ascending=False).head(10)
    plt.figure(figsize=(10, 5))
    sns.barplot(data=top10, x='total_orders', y='product_name',
                hue='product_name', palette='Blues_r', legend=False)
    plt.title('Top 10 Best-Selling Products')
    plt.xlabel('Total Orders')
    plt.ylabel('')
    plt.tight_layout()
    plt.savefig(f'{OUT}/top10_products.png', dpi=150)
    plt.close()
    print('    top10_products.png saved')

    hourly = orders_full.groupby('order_hour_of_day')['order_id'].count().reset_index()
    hourly.columns = ['hour', 'total_orders']
    plt.figure(figsize=(10, 4))
    sns.lineplot(data=hourly, x='hour', y='total_orders', marker='o', color='steelblue')
    plt.title('Orders by Hour of Day')
    plt.xlabel('Hour (0 = midnight, 10 = 10 AM)')
    plt.ylabel('Total Orders')
    plt.xticks(range(0, 24))
    plt.tight_layout()
    plt.savefig(f'{OUT}/hourly_orders.png', dpi=150)
    plt.close()
    print('    hourly_orders.png saved')

    dow = orders_full.groupby('order_dow')['order_id'].count().reset_index()
    dow.columns = ['day_of_week', 'total_orders']
    dow_labels = {0:'Sun', 1:'Mon', 2:'Tue', 3:'Wed', 4:'Thu', 5:'Fri', 6:'Sat'}
    dow['day_name'] = dow['day_of_week'].map(dow_labels)
    plt.figure(figsize=(8, 4))
    sns.barplot(data=dow, x='day_name', y='total_orders',
                hue='day_name', palette='Blues_r', legend=False)
    plt.title('Orders by Day of Week')
    plt.xlabel('Day')
    plt.ylabel('Total Orders')
    plt.tight_layout()
    plt.savefig(f'{OUT}/daily_orders.png', dpi=150)
    plt.close()
    print('    daily_orders.png saved')

    fi = pd.Series(
        xgb_model.feature_importances_, index=features
    ).sort_values(ascending=False)
    plt.figure(figsize=(8, 4))
    fi.plot(kind='barh', color='steelblue')
    plt.title('XGBoost - Feature Importance')
    plt.xlabel('Importance score')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(f'{OUT}/feature_importance.png', dpi=150)
    plt.close()
    print('    feature_importance.png saved')

    from sklearn.metrics import classification_report as cr
    lr_recall  = round(float(cr(y_test_store, lr_preds_store,  output_dict=True)['1']['recall']), 4)
    xgb_recall = round(float(cr(y_test_store, xgb_preds_store, output_dict=True)['1']['recall']), 4)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    metrics_list = ['Accuracy', 'AUC-ROC', 'Recall (reorders)']
    lr_vals  = [lr_acc,  lr_auc,  lr_recall]
    xgb_vals = [xgb_acc, xgb_auc, xgb_recall]
    for i, metric in enumerate(metrics_list):
        axes[i].bar(['LR', 'XGBoost'], [lr_vals[i], xgb_vals[i]],
                    color=['#B5D4F4', '#1D9E75'], edgecolor='none', width=0.5)
        axes[i].set_title(metric)
        axes[i].set_ylim(0, 1)
        for j, val in enumerate([lr_vals[i], xgb_vals[i]]):
            axes[i].text(j, val + 0.02, f'{val:.3f}', ha='center',
                         fontsize=10, fontweight='bold')
    plt.suptitle('Logistic Regression vs XGBoost', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUT}/model_comparison.png', dpi=150)
    plt.close()
    print('    model_comparison.png saved')

    del orders_full, order_products_full, product_demand
    gc.collect()

    with open(f'{MDL}/xgb_model_{today}.pkl', 'wb') as f:
        pickle.dump({'model': xgb_model, 'scaler': scaler, 'features': features}, f)
    with open(f'{MDL}/latest_model.pkl', 'wb') as f:
        pickle.dump({'model': xgb_model, 'scaler': scaler, 'features': features}, f)
    print(f'  Model saved: xgb_model_{today}.pkl')

    tier_counts = inventory['priority_tier'].value_counts()
    top5_inv    = inventory.head(5)[['product_name', 'total_orders', 'final_priority', 'priority_tier']].to_dict('records')
    top5_prod   = top_products.head(5)[['product_name', 'total_orders', 'reorder_rate']].to_dict('records')

    print('\nPIPELINE COMPLETE!')
    print(f'  XGBoost AUC-ROC: {xgb_auc}')
    print(f'  Products scored: {dataset["product_id"].nunique():,}')

    return {
        'status':             'success',
        'date':               today,
        'lr_accuracy':        lr_acc,
        'lr_auc_roc':         lr_auc,
        'xgb_accuracy':       xgb_acc,
        'xgb_auc_roc':        xgb_auc,
        'feature_importance': feat_importance,
        'products_scored':    int(dataset['product_id'].nunique()),
        'users_scored':       int(dataset['user_id'].nunique()),
        'avg_reorder_prob':   round(float(dataset['reorder_probability'].mean()), 4),
        'high_reorder_count': int((dataset['reorder_probability'] > 0.6).sum()),
        'inventory_tiers':    {str(k): int(v) for k, v in tier_counts.items()},
        'top_5_inventory':    top5_inv,
        'top_5_products':     top5_prod,
        'files_saved': {
            'predictions':  pred_file,
            'inventory':    inv_file,
            'top_products': top_file
        }
    }


print('ML function defined and ready!')

ML function defined and ready!


In [ ]:
# ============================================================
# CELL 5 — Optional manual test
# Uncomment to test cleaning or ML manually before n8n
# ============================================================

# Test cleaning:
# result = run_cleaning()
# print(json.dumps(result, indent=2))

# Test ML:
# result = run_ml()
# print(json.dumps(result, indent=2, default=str))

print('Cell 5 ready — uncomment above to test manually')

Cell 5 ready — uncomment above to test manually


In [ ]:
# ============================================================
# CELL 6 — Combined Flask server + ngrok
# ONE server for BOTH cleaning and ML endpoints
# RUN LAST — KEEP THIS CELL RUNNING
#
# Endpoints:
#   GET  /ping             → health check
#   POST /run-cleaning     → trigger cleaning (returns immediately)
#   GET  /cleaning-status  → poll cleaning result
#   POST /run-pipeline     → trigger ML (returns immediately)
#   GET  /status           → poll ML result
#
# n8n workflow:
#   Node 1: POST /run-cleaning    (timeout: 30000)
#   Node 2: Wait 8 mins
#   Node 3: GET  /cleaning-status (timeout: 30000)
#   Node 4: IF status == success
#   Node 5: POST /run-pipeline    (timeout: 30000)
#   Node 6: Wait 20 mins (ML takes 15-20 mins total)
#   Node 7: GET  /status          (timeout: 30000)
#   Node 8: IF status == success → Gemini → Email
# ============================================================

from flask   import Flask, jsonify, request as freq
from pyngrok import ngrok as pyngrok
import threading, time

# ── Paste your ngrok auth token here ─────────────────────────
pyngrok.kill()
pyngrok.set_auth_token('YOUR_NGROK_AUTH_TOKEN_HERE')

app = Flask(__name__)

# ── Shared job state for cleaning ────────────────────────────
clean_job = {
    'running':    False,
    'result':     None,
    'error':      None,
    'started_at': None,
    'done_at':    None,
    'step':       'idle'
}

# ── Shared job state for ML ───────────────────────────────────
ml_job = {
    'running':    False,
    'result':     None,
    'error':      None,
    'started_at': None,
    'done_at':    None,
    'step':       'idle'
}

# ── JSON encoder ─────────────────────────────────────────────
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        if hasattr(obj, 'item'):         return obj.item()
        return super().default(obj)

def safe_json(data, status=200):
    return app.response_class(
        response=json.dumps(data, cls=NumpyEncoder),
        status=status,
        mimetype='application/json'
    )

# ── Background workers ────────────────────────────────────────
def cleaning_worker():
    clean_job['running']    = True
    clean_job['result']     = None
    clean_job['error']      = None
    clean_job['started_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
    clean_job['done_at']    = None
    clean_job['step']       = 'starting'
    try:
        print('\n' + '='*50)
        print('Cleaning triggered by n8n!')
        clean_job['step']   = 'running run_cleaning()'
        result              = run_cleaning()
        clean_job['result'] = result
        clean_job['step']   = 'complete'
        clean_job['done_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
        print('Cleaning complete!')
        print('='*50)
    except Exception as e:
        import traceback
        clean_job['error']   = f'{str(e)}\n\n{traceback.format_exc()}'
        clean_job['step']    = 'error'
        clean_job['done_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
        print(f'\nCleaning ERROR: {e}')
    finally:
        clean_job['running'] = False


def pipeline_worker():
    ml_job['running']    = True
    ml_job['result']     = None
    ml_job['error']      = None
    ml_job['started_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
    ml_job['done_at']    = None
    ml_job['step']       = 'starting'
    try:
        print('\n' + '='*50)
        print('ML pipeline triggered by n8n!')
        ml_job['step']   = 'running run_ml()'
        result           = run_ml()
        ml_job['result'] = result
        ml_job['step']   = 'complete'
        ml_job['done_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
        print(f'ML pipeline complete! AUC-ROC: {result["xgb_auc_roc"]}')
        print('='*50)
    except Exception as e:
        import traceback
        ml_job['error']   = f'{str(e)}\n\n{traceback.format_exc()}'
        ml_job['step']    = 'error'
        ml_job['done_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
        print(f'\nML ERROR: {e}')
    finally:
        ml_job['running'] = False


# ── ENDPOINT: Health check ────────────────────────────────────
@app.route('/ping', methods=['GET'])
def ping():
    return safe_json({
        'status':         'alive',
        'cleaning_step':  clean_job['step'],
        'ml_step':        ml_job['step'],
        'message':        'Combined pipeline server is running!'
    })


# ── ENDPOINT: Trigger cleaning ────────────────────────────────
@app.route('/run-cleaning', methods=['POST'])
def run_cleaning_endpoint():
    if clean_job['running']:
        return safe_json({
            'status':  'busy',
            'message': 'Cleaning already running — poll /cleaning-status'
        }, status=429)
    threading.Thread(target=cleaning_worker, daemon=True).start()
    return safe_json({
        'status':     'started',
        'message':    'Cleaning started. Poll /cleaning-status for result.',
        'started_at': time.strftime('%Y-%m-%d %H:%M:%S')
    })


# ── ENDPOINT: Cleaning status ─────────────────────────────────
@app.route('/cleaning-status', methods=['GET'])
def cleaning_status():
    if clean_job['running']:
        elapsed = ''
        if clean_job['started_at']:
            from datetime import datetime
            secs    = int((datetime.now() - datetime.strptime(
                clean_job['started_at'], '%Y-%m-%d %H:%M:%S'
            )).total_seconds())
            elapsed = f'{secs//60}m {secs%60}s'
        return safe_json({'status': 'running', 'step': clean_job['step'], 'elapsed': elapsed})
    elif clean_job['error']:
        return safe_json({'status': 'error', 'message': clean_job['error']}, status=500)
    elif clean_job['result']:
        return safe_json({
            **clean_job['result'],
            'started_at': clean_job['started_at'],
            'done_at':    clean_job['done_at']
        })
    return safe_json({'status': 'idle', 'message': 'No cleaning has run yet.'})


# ── ENDPOINT: Trigger ML ──────────────────────────────────────
@app.route('/run-pipeline', methods=['POST'])
def run_pipeline():
    if ml_job['running']:
        return safe_json({
            'status':  'busy',
            'message': 'ML already running — poll /status'
        }, status=429)
    threading.Thread(target=pipeline_worker, daemon=True).start()
    return safe_json({
        'status':     'started',
        'message':    'ML pipeline started. Poll /status for result.',
        'started_at': time.strftime('%Y-%m-%d %H:%M:%S')
    })


# ── ENDPOINT: ML status ───────────────────────────────────────
@app.route('/status', methods=['GET'])
def status():
    if ml_job['running']:
        elapsed = ''
        if ml_job['started_at']:
            from datetime import datetime
            secs    = int((datetime.now() - datetime.strptime(
                ml_job['started_at'], '%Y-%m-%d %H:%M:%S'
            )).total_seconds())
            elapsed = f'{secs//60}m {secs%60}s'
        return safe_json({'status': 'running', 'step': ml_job['step'], 'elapsed': elapsed})
    elif ml_job['error']:
        return safe_json({'status': 'error', 'message': ml_job['error']}, status=500)
    elif ml_job['result']:
        return safe_json({
            **ml_job['result'],
            'started_at': ml_job['started_at'],
            'done_at':    ml_job['done_at']
        })
    return safe_json({'status': 'idle', 'message': 'No ML pipeline has run yet.'})


# ── Start Flask then ngrok ────────────────────────────────────
flask_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False)
)
flask_thread.daemon = True
flask_thread.start()

time.sleep(3)

# Static domain (recommended — URL never changes):
# tunnel = pyngrok.connect(5000, domain='your-domain.ngrok-free.app')
tunnel = pyngrok.connect(5000)
url    = tunnel.public_url

print('\n' + '='*60)
print('COMBINED PIPELINE SERVER IS LIVE!')
print('='*60)
print(f'\n  Health:          GET  {url}/ping')
print(f'  Trigger clean:   POST {url}/run-cleaning')
print(f'  Cleaning result: GET  {url}/cleaning-status')
print(f'  Trigger ML:      POST {url}/run-pipeline')
print(f'  ML result:       GET  {url}/status')
print('\n  Paste the same base URL into ALL n8n HTTP Request nodes')
print('  KEEP THIS TAB OPEN!')
print('='*60)

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.



COMBINED PIPELINE SERVER IS LIVE!

  Health:          GET  https://superaqual-jonelle-nonspatially.ngrok-free.dev/ping
  Trigger clean:   POST https://superaqual-jonelle-nonspatially.ngrok-free.dev/run-cleaning
  Cleaning result: GET  https://superaqual-jonelle-nonspatially.ngrok-free.dev/cleaning-status
  Trigger ML:      POST https://superaqual-jonelle-nonspatially.ngrok-free.dev/run-pipeline
  ML result:       GET  https://superaqual-jonelle-nonspatially.ngrok-free.dev/status

  Paste the same base URL into ALL n8n HTTP Request nodes
  KEEP THIS TAB OPEN!


In [ ]:
# ============================================================
# CELL 7 — Keep-alive
# Prevents Colab from disconnecting after 90 minutes
# ============================================================

from IPython.display import display, Javascript
display(Javascript('''
function keepAlive() {
    var btn = document.querySelector("#top-toolbar > colab-connect-button");
    if (btn) {
        var inner = btn.shadowRoot.querySelector("#connect");
        if (inner) inner.click();
    }
}
setInterval(keepAlive, 60000);
console.log("Keep-alive started");
'''))
print('Keep-alive running — Colab will not disconnect')

<IPython.core.display.Javascript object>

Keep-alive running — Colab will not disconnect
